In [ ]:
# @title  CIVITAI FILE NAME CHECKER
import requests

# Example Model Version ID from the URL (1450053)
model_version_id = "1828803"
api_url = f"https://civitai.com/api/v1/model-versions/{model_version_id}"

data = requests.get(api_url).json()

# Grab the primary file's name
for file in data.get('files', []):
    print(f"Official Filename: {file['name']}")

In [ ]:
# @title 🗑️ COMFYUI BATCH DELETER
import os
import shutil

# ============================================================
# ⚙️ SELECT WHAT TO DELETE
# Set any option to True to delete its contents/folder.
# ============================================================

# 1. Clean individual file types (Keeps the folder, removes files)
CLEAR_LORAS = False
CLEAR_CHECKPOINTS = False
CLEAR_CONTROLNETS = False
CLEAR_VAES = False
CLEAR_CLIPS = False
CLEAR_UPSCALE_MODELS = False

# 2. Clean generated outputs (Outputs, Temp caches, Logs)
CLEAR_OUTPUT_IMAGES = False
CLEAR_TEMP_CACHE = False

# 3. Nuclear options (Deletes entire directories)
REMOVE_ALL_CUSTOM_NODES = False
WIPE_ENTIRE_COMFYUI = False  # ⚠️ Wipes everything including ComfyUI installation

# 4. Delete specific files by name (e.g., ["corrupt_model.safetensors", "bad_lora.safetensors"])
SPECIFIC_FILES_TO_DELETE = [
    # "example_file_name.safetensors",
]


# ============================================================
# 🚀 DELETER ENGINE
# ============================================================
BASE_DIR = "/content/ComfyUI"
MODELS_DIR = os.path.join(BASE_DIR, "models")

print("="*60)
print("🗑️ COMFYUI BATCH DELETER INITIALIZED")
print("="*60)

def clear_folder_contents(folder_path, label):
    """Deletes all files inside a directory without removing the directory itself."""
    if not os.path.exists(folder_path):
        print(f"⏩ {label} directory does not exist. Skipping.")
        return

    count = 0
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
                count += 1
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
                count += 1
        except Exception as e:
            print(f"❌ Failed to delete {filename}: {e}")

    print(f"✅ Cleared {label}: {count} items removed.")

# --- Execute Folder Cleanups ---
if CLEAR_LORAS:
    clear_folder_contents(os.path.join(MODELS_DIR, "loras"), "LoRAs")

if CLEAR_CHECKPOINTS:
    clear_folder_contents(os.path.join(MODELS_DIR, "checkpoints"), "Checkpoints")

if CLEAR_CONTROLNETS:
    clear_folder_contents(os.path.join(MODELS_DIR, "controlnet"), "ControlNets")

if CLEAR_VAES:
    clear_folder_contents(os.path.join(MODELS_DIR, "vae"), "VAEs")

if CLEAR_CLIPS:
    clear_folder_contents(os.path.join(MODELS_DIR, "clip"), "CLIPs")

if CLEAR_UPSCALE_MODELS:
    clear_folder_contents(os.path.join(MODELS_DIR, "upscale_models"), "Upscalers")

if CLEAR_OUTPUT_IMAGES:
    clear_folder_contents(os.path.join(BASE_DIR, "output"), "Outputs")

if CLEAR_TEMP_CACHE:
    clear_folder_contents(os.path.join(BASE_DIR, "temp"), "Temp Cache")

if REMOVE_ALL_CUSTOM_NODES:
    clear_folder_contents(os.path.join(BASE_DIR, "custom_nodes"), "Custom Nodes")

# --- Execute Specific File Deletions ---
if SPECIFIC_FILES_TO_DELETE:
    print("\n🔍 Searching for specific target files...")
    deleted_specifics = 0
    for root, _, files in os.walk(BASE_DIR):
        for file in files:
            if file in SPECIFIC_FILES_TO_DELETE:
                full_path = os.path.join(root, file)
                os.remove(full_path)
                print(f"✅ Deleted specific file: {file}")
                deleted_specifics += 1
    if deleted_specifics == 0:
        print("⏩ No matching specific files found.")

# --- Execute Nuclear Wipe ---
if WIPE_ENTIRE_COMFYUI:
    if os.path.exists(BASE_DIR):
        print("\n⚠️ Wiping entire ComfyUI directory...")
        shutil.rmtree(BASE_DIR)
        print("💥 ComfyUI has been completely deleted.")

print("\n" + "="*60)
print("🎉 DELETER OPERATION COMPLETE")
print("="*60)

In [1]:
# @title 📥 COMFYUI ALL-IN-ONE MASTER DOWNLOADER
import os
import re
import getpass
import subprocess
import requests
from IPython.display import clear_output
from tqdm import tqdm

# ============================================================
# 📥 USER INPUTS: PLACE YOUR DOWNLOAD URLS HERE
# ============================================================

LORAS = [
     #///ILLUSTRIOUS///

    #Brazilian Miku LoRa | Illustrious /// brazilianmiku.safetensors
    "https://civitai.com/api/download/models/1450053?fileId=1351324",
    #[IllustriousXL v0.1] Brazilian Miku | Vocaloid /// vocaloid_brazilianmiku_illustriousXL.safetensors
    "https://civitai.com/api/download/models/1109601?fileId=1014519",

    #Illustrious Style Pack-IFL /// IFL_v1.0_IL.safetensors
    "https://civitai.com/api/download/models/2211883?fileId=2104890",
    #Illustrious Style Pack-LMB v2 /// LMB_style_v2.2_IL.safetensors
    "https://civitai.com/api/download/models/2123977?fileId=2018066",
    #Illustrious Style Pack-MSS v2 /// MSS_v2_IL.safetensors
    "https://civitai.com/api/download/models/1942096?fileId=1839728",
    #Illustrious Style Pack-PHM v3 /// PHM_style_IL_v3.3.safetensors
    "https://civitai.com/api/download/models/1570070?fileId=1470025",
    #T-Rex Studio V2 NEW!!- Hentai +18 - | STYLE | PONY XL | Illustrious XL | - COMMISSION - by YeiyeiArt /// ATRex_style-12V2Rev.safetensors
    "https://civitai.com/api/download/models/1804885?fileId=1705538",

]

CHECKPOINTS = [
    # MeinaMix (Anime/Illustration model) - ~2GB
    #"https://civitai.com/api/download/models/119057",

    #MeinaHentai
    #"https://civitai.com/api/download/models/948699?fileId=855570",

    # SDPose Wholebody - ~1.9GB
    #"https://huggingface.co/Comfy-Org/SDPose/resolve/main/checkpoints/sdpose_wholebody_fp16.safetensors?download=true",

    # Illustrious SDXL (Large model - ~6.5GB)
    "https://civitai.com/api/download/models/2883731",

    #PornWorks Anime Desire ● NSFW Anime & Hentai SDXL/Pony Chekpoint /// pornworksAnimeDesireNSFW_illustrious.safetensors
    "https://civitai.com/api/download/models/2275504?fileId=2167430",

    #Nova Anime XL /// novaAnimeXL_ilV190.safetensors
    "https://civitai.com/api/download/models/2940478?fileId=2819621",

    #Ultimate Hentai Anime RX - ( T-Rex ) - Anime ScreenShot - | CHECKPOINT - Illustrious XL | - by YeiyeiArt /// ultimateHentaiAnimeRXTRexAnime_rxV1.safetensors
    "https://civitai.com/api/download/models/1828803?fileId=1729137",

]

CONTROLNETS = [
    # OpenPose SD15
    #"https://civitai.com/api/download/models/537364?fileId=453956",

    # OpenPose SDXL
    #"https://huggingface.co/dimitribarbot/controlnet-openpose-sdxl-1.0-safetensors/resolve/main/diffusion_pytorch_model.safetensors?download=true",

    # Canny SD15
    #"https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.pth",

    # Depth SD15
    #"https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11f1p_sd15_depth.pth",

    # Xinsir ControlNet Union ProMax (All-in-one Pose, Canny, Depth for SDXL/Illustrious)
    "https://huggingface.co/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model_promax.safetensors?download=true",
]

VAES = [
    # SD 1.5 VAE - ~300MB
    #"https://civitai.com/api/download/models/311162?fileId=262135",

    # SDXL VAE - ~300MB
    "https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors",

]

CLIPS = [
    # Standalone CLIP text encoders (if required by custom workflows)

    # OpenAI CLIP Large
    #"https://huggingface.co/openai/clip-vit-large-patch14/resolve/main/model.safetensors",

]

UPSCALE_MODELS = [
    # 4x-AnimeSharp
    "https://civitai.com/api/download/models/1140894?fileId=1045879",

    # 4x_foolhardy_Remacri
    "https://huggingface.co/LyliaEngine/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.safetensors?download=true",

    # RealESRGAN
    #"https://civitai.com/api/download/models/164898?fileId=124730",

]

CUSTOM_NODES = [
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git",
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git",
    "https://github.com/ltdrdata/ComfyUI-Inspire-Pack.git",
    "https://github.com/cubiq/ComfyUI_IPAdapter_plus.git",
    "https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/cubiq/ComfyUI_essentials.git",
    "https://github.com/chflame163/ComfyUI_LayerStyle.git",
    "https://github.com/Fannovel16/comfyui_controlnet_aux.git",
    "https://github.com/storyicon/comfyui_segment_anything.git",
    "https://github.com/WASasquatch/was-node-suite-comfyui.git",
]


# ============================================================
# ⚙️ SYSTEM & DIRECTORY INITIALIZATION
# ============================================================
clear_output(wait=True)
print("="*60)
print("📥 COMFYUI ALL-IN-ONE MASTER DOWNLOADER")
print("="*60)

BASE_DIR = "/content/ComfyUI"
MODELS_DIR = os.path.join(BASE_DIR, "models")
CUSTOM_NODES_DIR = os.path.join(BASE_DIR, "custom_nodes")

# Map download lists to target ComfyUI directories
DOWNLOAD_MAP = {
    os.path.join(MODELS_DIR, "loras"): LORAS,
    os.path.join(MODELS_DIR, "checkpoints"): CHECKPOINTS,
    os.path.join(MODELS_DIR, "controlnet"): CONTROLNETS,
    os.path.join(MODELS_DIR, "vae"): VAES,
    os.path.join(MODELS_DIR, "clip"): CLIPS,
    os.path.join(MODELS_DIR, "upscale_models"): UPSCALE_MODELS,
    CUSTOM_NODES_DIR: CUSTOM_NODES,
}

# Create required directories
for target_dir in DOWNLOAD_MAP.keys():
    os.makedirs(target_dir, exist_ok=True)

print("✅ Directories configured.")

# Install fast downloader engines (aria2 and tqdm)
print("📦 Checking download engines (aria2 & tqdm)...")
subprocess.run("apt-get update -qq && apt-get install aria2 -y -qq", shell=True)
subprocess.run("pip install tqdm requests -q", shell=True)
print("✅ Downloader engines ready.")


# ============================================================
# 🔑 API & AUTHENTICATION SETUP
# ============================================================
print("\n" + "="*60)
print("🔑 API & AUTHENTICATION SETUP")
print("="*60)
print("ℹ️ Press Enter to skip if downloading public models only.\n")

civitai_key = getpass.getpass("Enter Civitai API Key (optional): ").strip() or None
hf_token = getpass.getpass("Enter HuggingFace Token (optional): ").strip() or None


# ============================================================
# 🛠️ HELPER FUNCTIONS
# ============================================================

def resolve_civitai_filename(url):
    """Attempts to fetch the exact filename from Civitai API or HTTP headers."""
    headers = {}
    if civitai_key:
        headers["Authorization"] = f"Bearer {civitai_key}"

    try:
        # Check if URL contains model version ID
        match = re.search(r'/models/(\d+)', url)
        if match and "civitai.com/api/download/models/" in url:
            model_version_id = match.group(1)
            api_url = f"https://civitai.com/api/v1/model-versions/{model_version_id}"
            resp = requests.get(api_url, headers=headers, timeout=10)
            if resp.status_code == 200:
                data = resp.json()
                files = data.get("files", [])
                if files:
                    # Return primary file name or first available file name
                    for file_info in files:
                        if file_info.get("primary", False):
                            return file_info.get("name")
                    return files[0].get("name")
    except Exception:
        pass
    return None

def get_filename_from_url(url):
    """Extract clean filename or fetch real name from API/Redirect headers."""
    # 1. URL contains explicit filename parameter
    if "filename=" in url:
        match = re.search(r'filename=([^&]+)', url)
        if match:
            return match.group(1)

    # 2. Try fetching true name from Civitai API
    if "civitai.com/api/download/models/" in url:
        api_filename = resolve_civitai_filename(url)
        if api_filename:
            return api_filename

    # 3. Fallback: Parse URL path
    clean_url = url.split('?')[0].rstrip('/')
    filename = clean_url.split('/')[-1]

    # If the filename ends with a raw numeric ID, attach .safetensors
    if filename.isdigit():
        filename += ".safetensors"

    return filename

def is_file_valid(filepath, min_size_mb=1):
    """Checks file existence, size threshold, and screens out HTML error pages."""
    if not os.path.exists(filepath):
        return False

    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    if size_mb < min_size_mb:
        return False

    try:
        with open(filepath, 'rb') as f:
            header = f.read(100)
            if header.startswith(b'<html') or header.startswith(b'<!DOCTYPE'):
                return False
    except Exception:
        return False

    return True

def download_file(url, output_dir, headers=None):
    """Downloads model using aria2c multi-connection with requests fallback."""
    filename = get_filename_from_url(url)
    output_path = os.path.join(output_dir, filename)

    # 1. Skip if already downloaded and valid
    if is_file_valid(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"⏩ Skipping {filename} (Already exists: {size_mb:.1f} MB)")
        return True, "skipped"

    # 2. Clean corrupt file if present
    if os.path.exists(output_path):
        os.remove(output_path)

    print(f"\n⬇️ Downloading: {filename}")

    # Build auth URL/headers
    download_url = url
    auth_headers = headers or {}

    if "civitai.com" in url and civitai_key:
        delimiter = "&" if "?" in url else "?"
        download_url = f"{url}{delimiter}token={civitai_key}"
    elif "huggingface.co" in url and hf_token:
        auth_headers["Authorization"] = f"Bearer {hf_token}"

    # 3. Primary attempt: aria2c (Fast multi-connection download)
    cmd = [
        "aria2c",
        "--console-log-level=error",
        "-c", "-x", "16", "-s", "16",
        "-k", "1M", "--timeout=60",
        "--max-tries=3",
        "--content-disposition",
        "-d", output_dir,
        "-o", filename,
        download_url
    ]

    for k, v in auth_headers.items():
        cmd.extend(["--header", f"{k}: {v}"])

    subprocess.run(cmd)

    # Catch case where aria2 saved file with content-disposition name
    if not os.path.exists(output_path):
        # Look for files without extensions or pure numbers and fix them
        for f in os.listdir(output_dir):
            if f.isdigit():
                old_p = os.path.join(output_dir, f)
                new_p = old_p + ".safetensors"
                os.rename(old_p, new_p)

    if is_file_valid(output_path):
        size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"✅ Downloaded successfully ({size_mb:.1f} MB)")
        return True, "downloaded"

    # 4. Fallback attempt: requests
    print("⚠️ aria2 failed or output invalid. Trying HTTP fallback...")
    try:
        resp = requests.get(download_url, headers=auth_headers, stream=True, timeout=60)
        if resp.status_code == 200:
            total_size = int(resp.headers.get('content-length', 0))
            with open(output_path, "wb") as f:
                with tqdm(total=total_size, unit='B', unit_scale=True, desc=filename[:25]) as pbar:
                    for chunk in resp.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            pbar.update(len(chunk))

            if is_file_valid(output_path):
                size_mb = os.path.getsize(output_path) / (1024 * 1024)
                print(f"✅ Downloaded successfully via fallback ({size_mb:.1f} MB)")
                return True, "downloaded"
    except Exception as e:
        print(f"❌ Fallback failed: {e}")

    if os.path.exists(output_path):
        os.remove(output_path)
    print(f"❌ Failed to download {filename}")
    return False, "failed"


# ============================================================
# 🚀 EXECUTION ENGINE
# ============================================================
print("\n" + "="*60)
print("🚀 PROCESSING DOWNLOAD QUEUE")
print("="*60)

stats = {"downloaded": 0, "skipped": 0, "failed": 0}

for target_dir, urls in DOWNLOAD_MAP.items():
    category_name = os.path.basename(target_dir).upper()
    valid_urls = [u.strip() for u in urls if u and u.strip() and not u.strip().startswith("#")]

    if not valid_urls:
        continue

    print(f"\n📂 CATEGORY: {category_name}")
    print("-" * 40)

    for url in valid_urls:
        # Handle Git Custom Nodes
        if target_dir == CUSTOM_NODES_DIR:
            repo_name = url.rstrip("/").split("/")[-1].replace(".git", "")
            node_path = os.path.join(target_dir, repo_name)

            if os.path.exists(node_path):
                print(f"⏩ Custom node '{repo_name}' already installed. Skipping.")
                stats["skipped"] += 1
            else:
                print(f"📦 Cloning custom node: {repo_name}...")
                res = subprocess.run(f"git clone --depth 1 {url} {node_path}", shell=True, capture_output=True)
                if res.returncode == 0:
                    print("✅ Custom node installed.")
                    stats["downloaded"] += 1
                else:
                    print(f"❌ Failed to clone {repo_name}")
                    stats["failed"] += 1
            continue

        # Handle Standard Files
        success, state = download_file(url, target_dir)
        stats[state] += 1

print("\n" + "="*60)
print(f"🎉 BATCH COMPLETE: {stats['downloaded']} downloaded | {stats['skipped']} skipped | {stats['failed']} failed")
print("="*60)

📥 COMFYUI ALL-IN-ONE MASTER DOWNLOADER
✅ Directories configured.
📦 Checking download engines (aria2 & tqdm)...
✅ Downloader engines ready.

🔑 API & AUTHENTICATION SETUP
ℹ️ Press Enter to skip if downloading public models only.

Enter Civitai API Key (optional): ··········
Enter HuggingFace Token (optional): ··········

🚀 PROCESSING DOWNLOAD QUEUE

📂 CATEGORY: LORAS
----------------------------------------

⬇️ Downloading: brazilianmiku.safetensors
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


brazilianmiku.safetensors: 100%|██████████| 228M/228M [00:01<00:00, 161MB/s]


✅ Downloaded successfully via fallback (217.9 MB)

⬇️ Downloading: vocaloid_brazilianmiku_illustriousXL.safetensors
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


vocaloid_brazilianmiku_il: 100%|██████████| 57.5M/57.5M [00:00<00:00, 157MB/s]


✅ Downloaded successfully via fallback (54.8 MB)

⬇️ Downloading: IFL_v1.0_IL.safetensors
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


...

✅ Downloaded successfully via fallback (54.8 MB)

⬇️ Downloading: ATRex_style-12V2Rev.safetensors
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


ATRex_style-12V2Rev.safet: 100%|██████████| 114M/114M [00:03<00:00, 34.8MB/s] 


✅ Downloaded successfully via fallback (109.2 MB)

📂 CATEGORY: CHECKPOINTS
----------------------------------------

⬇️ Downloading: waiIllustriousSDXL_v170.safetensors
✅ Downloaded successfully (6616.6 MB)

⬇️ Downloading: pornworksAnimeDesireNSFW_illustrious.safetensors
✅ Downloaded successfully (6616.6 MB)

⬇️ Downloading: novaAnimeXL_ilV190.safetensors
✅ Downloaded successfully (6617.6 MB)

⬇️ Downloading: ultimateHentaiAnimeRXTRexAnime_rxV1.safetensors
✅ Downloaded successfully (6616.6 MB)

📂 CATEGORY: CONTROLNET
----------------------------------------

⬇️ Downloading: diffusion_pytorch_model_promax.safetensors
✅ Downloaded successfully (2396.9 MB)

📂 CATEGORY: VAE
----------------------------------------

⬇️ Downloading: sdxl_vae.safetensors
✅ Downloaded successfully (319.1 MB)

📂 CATEGORY: UPSCALE_MODELS
----------------------------------------

⬇️ Downloading: 4xAnimesharp_v10.zip
⚠️ aria2 failed or output invalid. Trying HTTP fallback...


4xAnimesharp_v10.zip: 100%|██████████| 30.9M/30.9M [00:00<00:00, 167MB/s]


✅ Downloaded successfully via fallback (29.5 MB)

⬇️ Downloading: 4x_foolhardy_Remacri.safetensors
✅ Downloaded successfully (63.8 MB)

📂 CATEGORY: CUSTOM_NODES
----------------------------------------
📦 Cloning custom node: ComfyUI-Manager...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-Advanced-ControlNet...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-Custom-Scripts...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-Impact-Pack...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-Inspire-Pack...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI_IPAdapter_plus...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-AnimateDiff-Evolved...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI-VideoHelperSuite...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI_essentials...
✅ Custom node installed.
📦 Cloning custom node: ComfyUI_LayerStyle...
✅ Custom node installed.
📦 Cloning custom node: comfyui_controlnet_aux...
✅ Custom node ins

In [ ]:
# @title COMFYUI SETUP SCRIPT
import getpass, os, requests, subprocess, time, socket, shutil
from IPython.display import clear_output

# ============================================================
# 🚀 COMFYUI V1 (STABLE PRODUCTION UI) LAUNCHER
# ============================================================
clear_output(wait=True)
print("="*60)
print("🚀 COMFYUI V1 LAUNCHER (STABLE PRODUCTION FRONTEND)")
print("="*60)

# ============================================================
# 📁 PATHS
# ============================================================
BASE_DIR = "/content/ComfyUI"
MODELS_DIR = os.path.join(BASE_DIR, "models")
CUSTOM_NODES_DIR = os.path.join(BASE_DIR, "custom_nodes")
TEMP_MODELS = "/content/temp_models_backup"
TEMP_NODES = "/content/temp_nodes_backup"

print("\n📁 Checking existing files...")

# ============================================================
# 💾 BACKUP & RESTORE FUNCTIONS
# ============================================================
def backup_models_and_nodes():
    needs_backup = False

    if os.path.exists(MODELS_DIR) and any(os.scandir(MODELS_DIR)):
        needs_backup = True
        print("✅ Models found (will preserve)")

    if os.path.exists(CUSTOM_NODES_DIR) and any(os.scandir(CUSTOM_NODES_DIR)):
        needs_backup = True
        print("✅ Custom nodes found (will preserve)")

    if not needs_backup:
        print("ℹ️ No existing models or nodes to backup")
        return False

    print("📦 Backing up models and custom nodes...")

    if os.path.exists(MODELS_DIR) and any(os.scandir(MODELS_DIR)):
        if os.path.exists(TEMP_MODELS):
            shutil.rmtree(TEMP_MODELS)
        shutil.move(MODELS_DIR, TEMP_MODELS)
        print("   ✅ Models backed up")

    if os.path.exists(CUSTOM_NODES_DIR) and any(os.scandir(CUSTOM_NODES_DIR)):
        if os.path.exists(TEMP_NODES):
            shutil.rmtree(TEMP_NODES)
        shutil.move(CUSTOM_NODES_DIR, TEMP_NODES)
        print("   ✅ Custom nodes backed up")

    return True

def restore_models_and_nodes():
    restored = False

    if os.path.exists(TEMP_MODELS):
        print("📦 Restoring models...")
        if os.path.exists(MODELS_DIR):
            shutil.rmtree(MODELS_DIR)
        shutil.move(TEMP_MODELS, MODELS_DIR)
        print("   ✅ Models restored")
        restored = True

    if os.path.exists(TEMP_NODES):
        print("📦 Restoring custom nodes...")
        if os.path.exists(CUSTOM_NODES_DIR):
            shutil.rmtree(CUSTOM_NODES_DIR)
        shutil.move(TEMP_NODES, CUSTOM_NODES_DIR)
        print("   ✅ Custom nodes restored")
        restored = True

    return restored

# ============================================================
# 🔧 INSTALL DEPENDENCIES
# ============================================================
print("\n📦 Installing dependencies...")

try:
    from pyngrok import ngrok
    print("✅ pyngrok already installed")
except ImportError:
    print("📦 Installing pyngrok...")
    subprocess.run("pip install pyngrok huggingface-hub -q", shell=True, capture_output=True)
    from pyngrok import ngrok
    print("✅ pyngrok installed")

# ============================================================
# 📥 COMFYUI SETUP
# ============================================================
print("\n📥 Setting up ComfyUI...")

needs_reinstall = False
if os.path.exists(BASE_DIR):
    if not os.path.exists(os.path.join(BASE_DIR, "main.py")):
        needs_reinstall = True
        print("⚠️ main.py missing - ComfyUI needs repair")
    else:
        print("✅ ComfyUI already installed")
else:
    needs_reinstall = True
    print("⬇️ ComfyUI not found - will install")

if needs_reinstall:
    backed_up = backup_models_and_nodes()

    if os.path.exists(BASE_DIR):
        print("🧹 Removing incomplete ComfyUI...")
        shutil.rmtree(BASE_DIR)
        print("✅ Removed")

    print("⬇️ Downloading latest ComfyUI repository...")
    result = subprocess.run(
        "git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI",
        shell=True,
        capture_output=True
    )
    if result.returncode == 0:
        print("✅ ComfyUI downloaded")
    else:
        print("❌ Failed to clone repository.")
        raise SystemExit("Clone error - check your network connection.")

    if backed_up:
        restore_models_and_nodes()
        print("✅ Models and nodes restored")

    print("📦 Installing ComfyUI requirements...")
    subprocess.run(f"pip install -r {BASE_DIR}/requirements.txt -q", shell=True)
    print("✅ Requirements installed")
else:
    print("🔄 Updating existing ComfyUI to latest version...")
    os.chdir(BASE_DIR)
    subprocess.run("git fetch --all", shell=True, capture_output=True)
    subprocess.run("git reset --hard origin/master", shell=True, capture_output=True)
    subprocess.run("git pull origin master", shell=True, capture_output=True)
    os.chdir("/content")
    print("✅ ComfyUI updated")

if not os.path.exists(os.path.join(BASE_DIR, "main.py")):
    print("❌ main.py missing! Installation failed.")
    raise SystemExit("ComfyUI installation failed.")

# ============================================================
# 🌐 NGROK SETUP
# ============================================================
print("\n" + "="*60)
print("🌐 NGROK SETUP")
print("="*60)

NGROK_AUTHTOKEN = getpass.getpass("🔑 Enter your ngrok Authtoken: ").strip()
if not NGROK_AUTHTOKEN:
    NGROK_AUTHTOKEN = input("⚠️ No token entered. Paste your ngrok token: ").strip()

ngrok.set_auth_token(NGROK_AUTHTOKEN)
print("✅ ngrok authenticated")

# ============================================================
# 🧹 CLEANUP OLD PROCESSES
# ============================================================
print("\n🧹 Cleaning up old processes...")
try:
    ngrok.kill()
    print("   ngrok killed")
except:
    pass

subprocess.run("pkill -f 'python main.py'", shell=True, capture_output=True)
subprocess.run("pkill -f 'python3 main.py'", shell=True, capture_output=True)
time.sleep(2)
print("✅ Cleanup complete")

# ============================================================
# 🚀 LAUNCH NGROK & COMFYUI WITH STABLE V1 UI
# ============================================================
print("\n🚀 Launching ngrok tunnel...")
tunnel = ngrok.connect(8188, "http")
public_url = tunnel.public_url.replace("http://", "https://")
print(f"✅ ngrok tunnel created: {public_url}")

print("\n🚀 Launching ComfyUI with Stable V1 Frontend...")

# Passes @stable to serve production V1 UI without Nightly bugs
comfyui_process = subprocess.Popen(
    [
        "python3", "main.py",
        "--listen", "127.0.0.1",
        "--port", "8188",
        "--lowvram",
        "--front-end-version", "Comfy-Org/ComfyUI_frontend@stable"
    ],
    cwd=BASE_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print(f"✅ ComfyUI launched (PID: {comfyui_process.pid})")

# ============================================================
# ⏳ HEALTH CHECK
# ============================================================
print("\n⏳ Waiting for ComfyUI to start...")
print(" Waiting", end="")

max_attempts = 60
started = False

for i in range(max_attempts):
    time.sleep(2)
    print(".", end="", flush=True)

    if comfyui_process.poll() is not None:
        print("\n❌ ComfyUI crashed during startup!")
        print("-" * 40)
        output = comfyui_process.stdout.read()
        if output:
            lines = output.split('\n')
            print('\n'.join(lines[-30:]))
        print("-" * 40)
        break

    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    result = sock.connect_ex(('127.0.0.1', 8188))
    sock.close()

    if result == 0:
        print(" ✅ READY!")
        started = True
        break
else:
    print("\n⚠️ ComfyUI took too long to start.")
    comfyui_process.terminate()

# ============================================================
# ✅ FINAL OUTPUT & KEEP-ALIVE LOOP
# ============================================================
clear_output(wait=True)

if started:
    print("="*60)
    print("✅ COMFYUI STABLE V1 IS RUNNING!")
    print("="*60)
    print(f"\n🔗 Access URL: {public_url}")
    print(f"⚠️  If you see an ngrok check screen, append: {public_url}?ngrok-browser-check=true")
    print("\n" + "="*60)
    print("⏳ Keep this cell running")
    print("🛑 Click STOP in Colab to terminate ComfyUI")
    print("="*60 + "\n")

    try:
        while True:
            if comfyui_process.poll() is not None:
                print("\n⚠️ ComfyUI process stopped.")
                break
            time.sleep(20)
    except KeyboardInterrupt:
        print("\n🛑 Shutting down...")
        try:
            ngrok.disconnect(public_url)
            ngrok.kill()
        except:
            pass
        comfyui_process.terminate()
        print("✅ Closed cleanly.")
else:
    print("="*60)
    print("❌ COMFYUI FAILED TO START")
    print("="*60)

✅ COMFYUI STABLE V1 IS RUNNING!

🔗 Access URL: https://helper-outfield-grunt.ngrok-free.dev
⚠️  If you see an ngrok check screen, append: https://helper-outfield-grunt.ngrok-free.dev?ngrok-browser-check=true

⏳ Keep this cell running
🛑 Click STOP in Colab to terminate ComfyUI

